# Bronze Layer: Extraction

**Target Tables:**
- **Read:** `bronze_general_search` (SQLite `scraping_database.db`)
- **Write:** `bronze_ad_links` / Raw HTML storage (SQLite / Parquet)

**Objective:**
This notebook takes the list of URLs discovered in the `discover.ipynb` notebook and fetches the detailed HTML/JSON data for each ad. It filters the target regions based on the metadata to avoid unnecessary requests and extracts the raw content of each advertisement, saving it to the Bronze layer without heavy transformations.

**To-Do:**
- Implement the detailed extraction logic for OLX.
- Implement rate-limiting, retries, and anti-blocking strategies.
- Decide on the storage format for raw HTML/JSON (e.g., Parquet or Blob storage).

In [1]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parents[2])
if project_root not in sys.path:
    sys.path.append(project_root)

from sqlalchemy import insert, select, update
from sqlalchemy.orm import Session
import polars as pl
from crawlers import OLXCrawler

# Importing from 'app' module
from app.services import OLXService
from app.config import db_engine
from app.models import GeneralSearch, InformationExtraction
from app.utils import read_search_locations_metadata

In [2]:
# List to store the pages' data
extraction_data_list = []

# List to store the selled/deleted products
not_available_products = []

In [3]:
# Region data for filtered results due to limits
brazil_data_location = (
    read_search_locations_metadata()
    .get('search_locations', {})
    .get('brazil', {})
)

stores = (
    brazil_data_location
    .get('stores', [])
)

regions = (
    brazil_data_location
    .get('regions', [])
)

full_name_list = [region.get('full_name') for region in regions]
abreviation_list = [region.get('abreviation') for region in regions]

In [4]:
# Reading values
with db_engine.connect() as connection:
    # Removing undesired regions
    stmt = select(GeneralSearch).where(GeneralSearch.region.in_(abreviation_list))
    df_olx = pl.read_database(query=stmt.where(GeneralSearch.store.is_('OLX')), connection=connection)


In [5]:
df_olx = df_olx.head(100)

In [ ]:
olx_crawler = OLXCrawler(base_data_list=extraction_data_list)
await olx_crawler.scrap_specific_information(
    df=df_olx,
    not_available_products=not_available_products
)

<coroutine object OLXCrawler.scrap_specific_information at 0x765d13004400>

In [7]:
# Creating dataframe
extraction_data_df = pl.DataFrame(extraction_data_list)
updated_df = pl.DataFrame(not_available_products)

if not extraction_data_df.is_empty():
    with Session(db_engine) as session:
        # Saving on sqlite
        session.execute(insert(InformationExtraction), extraction_data_df.to_dicts())
        session.commit()
    
    if not updated_df.is_empty():
        session.execute(update(GeneralSearch), updated_df.to_dicts())
        session.commit()
    else:
        print("No data found to update.")
else:
    print("No data found to insert.")

No data found to insert.
